In [3]:
import torch

# Problem (cross_entropy):  实现交叉熵损失 (1 分)
def cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    '''
    logits: (..., seq_len, vocab_size), 通常为 (batch_size, seq_len, vocab_size)
    targets: (..., seq_len) (B, S)
    '''
    # 交叉熵损失的计算：-log softmax(o)[target] = -(o[target] - max(o)) + log Σ exp(o - max(o))
    # 先减最大值再取 exp 避免上溢，log 与 exp 相互抵消

    # 1.默认不保留归约后的维度；2. max 返回的是元组 (最大值, 最大值对应的下标) 
    max_logits = logits.max(dim=-1)[0] # (B, S, V) -> (B, S)

    
    # max_logits.unsqueeze(-1) -> (B, S, 1); 等价于 max_logits[..., None] 
    # max_logits.unsqueeze(-1).squeeze(-1) -> (B, S); squeeze(-1)：删除最后维度，仅当该维度大小等于 1

    # 在最后一维收集目标标签对应的 logit 值: logits[..., targets]; logits[targets] 只能在第一维取元素，与需求不符
    target_logit = torch.gather(logits, -1, targets[..., None]).squeeze(-1) # (B, S, V), (B, S, 1) -> (B, S)

    term1 = -(target_logit - max_logits) # (B, S), (B, S)  -> (B, S) 

    exp_sum = torch.exp(logits - max_logits[..., None]).sum(dim=-1) # (B, S, V), (B, S, V) -> (B, S) 

    # 按照题目要求需要直接返回平均损失 loss.mean()，这里为了方便下面示例的比较，返回的是各个位置的损失
    return term1 + torch.log(exp_sum) # (B, S), (B, S) -> (B, S)


In [4]:
# 测试 cross_entropy
batch_size, seq_len, vocab_size = 2, 5, 1000

logits = torch.randn(batch_size, seq_len, vocab_size)          # 模型输出的未归一化分数 (B, S, V)
targets = torch.randint(0, vocab_size, (batch_size, seq_len))  # 目标 token 下标 (B, S)

loss = cross_entropy(logits, targets)
print(f"loss 形状：{loss.shape}")

# 与 PyTorch 官方实现对照，验证数值是否一致
import torch.nn.functional as F
ref = F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1), reduction="none").view(batch_size, seq_len)

# 与 F.cross_entropy 的误差
(loss - ref)



loss 形状：torch.Size([2, 5])


tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 4.7684e-07, 0.0000e+00, 0.0000e+00]])

In [ ]:
# 困惑度 perplexity：交叉熵用于训练足够了，评估模型时更习惯报告困惑度
# 可以理解为“平均意义下模型在多少个 token 之间犹豫”，数值越小越好
def cal_perplexity(loss: torch.Tensor) -> torch.Tensor:
    return torch.exp(loss.mean(dim=-1)) # (B, S) -> (B,)：先对序列维取平均，再取指数


In [ ]:
# 测试 cal_perplexity：基于上面的 loss 计算每个样本的困惑度
ppl = cal_perplexity(loss)      # (B, S) -> (B,)
print(f"困惑度形状：{ppl.shape}")
print(f"各样本困惑度：{ppl}")

# 整批数据的整体困惑度：先对所有位置取平均再取指数
print(f"整体困惑度：{torch.exp(loss.mean()).item():.4f}")
